# CMO emission ontology knowledge graph

Builds a typed Neo4j knowledge graph from `LuaHistory_2026-06-23.txt` with emitter type, kinematics, location, platform identity/variant hypotheses, and operator-country hypotheses.

In [ ]:
from pathlib import Path
from combat_id_calibration.cmo_observation_ingest import parse_observations, write_observations_jsonl, populate_observations_neo4j

LOG_PATH = Path("../LuaHistory_2026-06-23.txt")
observations = list(parse_observations(LOG_PATH.read_text(encoding="utf-8-sig", errors="replace").splitlines()))
len(observations), observations[0] if observations else None

In [ ]:
import pandas as pd
rows = [obs.__dict__ for obs in observations]
df = pd.DataFrame(rows)
df.head()

In [ ]:
summary = df.groupby(["emission_sensor_name", "emission_target_type"], dropna=False).size().reset_index(name="observations")
summary.sort_values("observations", ascending=False).head(20)

## Persist review artifact

The JSONL file is optional but useful for auditing before loading Neo4j.

In [ ]:
OUTPUT = Path("cmo_emission_observations_v2.jsonl")
write_observations_jsonl(observations, OUTPUT)
OUTPUT, OUTPUT.stat().st_size

## Load Neo4j

Set credentials before running this cell. The ontology creates nodes for `Observation`, `Emission`, `EmitterType`, `Kinematics`, `Location`, `Hypothesis`, `PlatformIdentity`, `OperatorCountry`, and supporting source/contact/sensor/platform nodes.

In [ ]:
NEO4J_URI = "bolt://localhost:7687"
NEO4J_USER = "neo4j"
NEO4J_PASSWORD = ""  # fill in locally
NEO4J_DATABASE = None

if NEO4J_PASSWORD:
    populate_observations_neo4j(observations, NEO4J_URI, NEO4J_USER, NEO4J_PASSWORD, NEO4J_DATABASE)
else:
    print("Set NEO4J_PASSWORD to ingest the graph.")

## Example Cypher queries

```cypher
MATCH (c:Contact)-[:HAS_HYPOTHESIS]->(h:Hypothesis)-[:IDENTIFIES_VARIANT]->(v:PlatformIdentity),
      (h)-[:HAS_OPERATOR_COUNTRY]->(oc:OperatorCountry),
      (h)-[:SUPPORTED_BY]->(o:Observation)-[:HAS_LOCATION]->(l:Location),
      (o)-[:HAS_KINEMATICS]->(k:Kinematics)
RETURN v.name AS variant, oc.name AS operator_country, l.latitude AS lat, l.longitude AS lon,
       k.heading AS heading, k.altitude AS altitude, k.speed AS speed
LIMIT 25;
```